In [1]:
import sys
sys.path.append('/Users/mariana/Documents/others_code/signature_survival')
sys.path.append('/Users/mariana/Documents/others_code/signature_survival/data_loader')

sys.path.append('/Users/mariana/Documents/col/lts4/survan')
sys.path.append('/Users/mariana/Documents/col/lts4/tdsurv/lib')

In [2]:
import numpy as np

In [3]:
from data_loader import load_NASA
paths, surv_labels, ddh_info_sup = load_NASA.load()
# paths = paths.clone()
n_samples, n_sampling_times, _ = paths.shape
sampling_times = paths[0, :, 0]
surv_times, surv_inds = surv_labels[:, 0], surv_labels[:, 1]

/Users/mariana/Documents/others_code/signature_survival/data_loader/load_NASA.py:34: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+', header=None, names=col_names)
/Users/mariana/Documents/others_code/signature_survival/data_loader/load_NASA.py:36: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+', header=None, names=col_names)
/Users/mariana/Documents/others_code/signature_survival/data_loader/load_NASA.py:38: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  dataset_train['tte'] = dataset_train.groupby(['id'])['times'].transform(max)
/Users/mariana/Documents/others_code/signature_survival/data_loader/load_NASA.py:40: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. T

In [4]:
paths.shape

(200, 363, 17)

In [5]:
surv_inds  # These are the labels indicating that the event did occur

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [6]:
# We take the opposite of surv_inds to get the censoring labels
cs = 1 - surv_inds
cs = cs.astype(int)

In [7]:
# surv_times are the times divided by 100. Not sure why they do that... we multiply by 100 to get the times:

ts = surv_times * 100
ts = ts.astype(int)

In [13]:
ts[ts>300]

array([312, 361, 340, 335, 302])

In [18]:
dummy_cs = np.zeros((len(ts), ))

In [19]:
from utils import get_targets_and_masks

# We use a dummy_cs with no censoring so that masks are correctly calculated. In this dataset measurements might stop before the
# horizon is reached despite the event not occuring.
tgt_lm, hws_lm, mask_lm = get_targets_and_masks(paths, ts, dummy_cs, landmark=True)

In [20]:
paths.shape

(200, 363, 17)

In [21]:
tgt_lm[0].sum()

191.0

In [43]:
tgt_lm[cs == 0].sum()

20525.0

In [39]:
# Multiply each entry of tgt_lm by 0 when cs == 1:
tgt_lm[cs == 1] *= 0

In [44]:
ts[cs==0].sum()

20525

In [36]:
ts[cs == 1][9]

191

In [24]:
tgt_lm[150].sum()

143.0

In [25]:
mask_lm[150][:ts[150]]

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]])

In [26]:
hws_lm[150][1].sum()

142

In [28]:
mask_lm

array([[[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, Fal

In [45]:
d = {
        "seqs": paths,
        "cs": cs,
        "ts": ts,
        "h_tgt": tgt_lm,
        "h_ws": hws_lm,
        "mask": mask_lm,
    }

In [46]:
# Store the data in a hdf5 file:
import h5py

with h5py.File('NASA.h5', 'w') as f:
    for k, v in d.items():
        f.create_dataset(k, data=v)

In [48]:
# Load the data to check that it was stored correctly:
with h5py.File('NASA.h5', 'r') as f:
    tgt = f['h_tgt'][:]

In [55]:
tgt[cs == 1].sum()

0.0

In [53]:
tgt = data['h_tgt'][:]

In [51]:
data = h5py.File('../../SurvanData/NASA.h5', 'r')

In [52]:
data.keys()

<KeysViewHDF5 ['cs', 'h_tgt', 'h_ws', 'mask', 'seqs', 'ts']>

In [9]:
# Change key name 'target' to 'h_tgt'
data.move('target', 'h_tgt')

# Save the changes in the same file
data.close()



ValueError: Unable to move link (no write intent on file)